# Series de Tiempo — Modelos Lineales

Este notebook cubre los modelos lineales clásicos para series de tiempo desde una perspectiva de procesos estocásticos.  
Quedan explícitamente fuera del alcance:
- Modelos de volatilidad variable: **ARCH, GARCH** y variantes
- Tendencias deterministas y modelos de descomposición
- Análisis espectral y densidad espectral
- Modelos de espacio de estados y filtro de Kalman

**Referencias generales:**
- Box, G., Jenkins, G., Reinsel, G. & Ljung, G. (2015). *Time Series Analysis: Forecasting and Control*. Wiley.
- Brockwell, P. & Davis, R. (1991). *Time Series: Theory and Methods*. Springer.
- Hamilton, J. (1994). *Time Series Analysis*. Princeton University Press.
- Shumway, R. & Stoffer, D. (2017). *Time Series Analysis and Its Applications*. Springer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import toeplitz
import warnings
warnings.filterwarnings('ignore')

# statsmodels para estimación y tests
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print("Librerías cargadas.")

---
# Bloque 0 — Series de tiempo como procesos estocásticos

En el análisis de procesos estocásticos general disponemos de **muchas realizaciones** del proceso y podemos estimar distribuciones marginales y conjuntas empíricamente.  
En series de tiempo tenemos **una sola trayectoria** — un único $\omega$ observado a lo largo del tiempo.

Esto plantea un problema fundamental: ¿cómo estimar la media $\mu = E[X_t]$ si solo tenemos un valor de $X_t$ para cada $t$?

La respuesta es la **ergodicidad**: bajo ciertas condiciones, los promedios temporales convergen a los promedios del ensemble:

$$\bar{X}_T = \frac{1}{T}\sum_{t=1}^T X_t \xrightarrow{T\to\infty} E[X_t] = \mu$$

Una condición suficiente (Brockwell & Davis, 1991, Prop. 7.1.1): $\sum_{h=-\infty}^\infty |\gamma(h)| < \infty$, es decir, que la autocovarianza sea absolutamente sumable.

In [ ]:
# Ilustración de ergodicidad: un proceso ergódico vs uno no ergódico
# Proceso ergódico: MA(1)
# Proceso NO ergódico: X_t = A + W_t, donde A ~ N(0,1) es fija para cada trayectoria

n_obs = 500
n_tray = 8
theta_ma = 0.7

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

medias_temporales_erg  = []
medias_temporales_nerg = []

for i in range(n_tray):
    color = plt.cm.tab10(i / n_tray)

    # Ergódico: MA(1)
    eps = np.random.normal(0, 1, n_obs + 1)
    ma1 = eps[1:] + theta_ma * eps[:-1]
    media_temp_erg = np.cumsum(ma1) / np.arange(1, n_obs + 1)
    axes[0, 0].plot(ma1, lw=0.8, alpha=0.6, color=color)
    axes[0, 1].plot(media_temp_erg, lw=1.5, alpha=0.8, color=color)
    medias_temporales_erg.append(ma1.mean())

    # No ergódico: X_t = A + W_t
    A = np.random.normal(0, 2)  # componente aleatoria fija por trayectoria
    x_nerg = A + np.random.normal(0, 0.3, n_obs)
    media_temp_nerg = np.cumsum(x_nerg) / np.arange(1, n_obs + 1)
    axes[1, 0].plot(x_nerg, lw=0.8, alpha=0.6, color=color)
    axes[1, 1].plot(media_temp_nerg, lw=1.5, alpha=0.8, color=color)
    medias_temporales_nerg.append(x_nerg.mean())

for ax in [axes[0, 1], axes[1, 1]]:
    ax.axhline(0, color='black', lw=2, ls='--', label='μ = 0  (verdadera)')
    ax.set_xlabel('T'); ax.set_ylabel('Media temporal')
    ax.legend(fontsize=9)

axes[0, 0].set_title('MA(1) — proceso ergódico\n(trayectorias)')
axes[0, 1].set_title('MA(1) — media temporal converge a μ')
axes[1, 0].set_title('$X_t = A + W_t$ — proceso NO ergódico\n(cada trayectoria tiene su propia media)')
axes[1, 1].set_title('$X_t = A + W_t$ — media temporal converge a A ≠ μ')

for ax in axes[:, 0]:
    ax.set_xlabel('t'); ax.set_ylabel('X_t')

fig.suptitle('Ergodicidad: una sola trayectoria es suficiente (o no)\n'
             'Ref: Brockwell & Davis (1991), Time Series: Theory and Methods, Prop. 7.1.1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Estimación de la función de autocovarianza

Con una sola trayectoria, el estimador natural de $\gamma(h)$ es:

$$\hat{\gamma}(h) = \frac{1}{T} \sum_{t=1}^{T-|h|} (X_t - \bar{X})(X_{t+|h|} - \bar{X})$$

Notar que se divide por $T$ y no por $T-|h|$: esto garantiza que la matriz de autocovarianza estimada sea semidefinida positiva (Brockwell & Davis, 1991, Prop. 7.2.1).  

El estimador es **sesgado** pero **consistente** bajo ergodicidad. Para lags grandes ($|h|$ cercano a $T$) la estimación es muy variable — regla práctica: usar $|h| \leq T/4$.

In [ ]:
# Efecto del tamaño muestral en la estimación de la ACF
# Proceso: AR(1) con phi=0.7
phi_ar = 0.7
max_lag = 20
lags = np.arange(max_lag + 1)

# ACF teórica: ρ(h) = phi^|h|
acf_teo = phi_ar ** lags

n_sizes = [50, 200, 1000]
n_rep = 200

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, n in zip(axes, n_sizes):
    acf_muestras = []
    for _ in range(n_rep):
        # Simular AR(1)
        x = np.zeros(n)
        eps = np.random.normal(0, 1, n)
        for t in range(1, n):
            x[t] = phi_ar * x[t-1] + eps[t]
        acf_m = acf(x, nlags=max_lag, fft=True)
        acf_muestras.append(acf_m)

    acf_muestras = np.array(acf_muestras)
    media_acf = acf_muestras.mean(axis=0)
    std_acf   = acf_muestras.std(axis=0)

    ax.fill_between(lags, media_acf - 2*std_acf, media_acf + 2*std_acf,
                    alpha=0.2, color='#2196F3', label='±2 desvíos (simulación)')
    ax.plot(lags, media_acf, 'o-', color='#2196F3', lw=2, markersize=4, label='Media empírica')
    ax.plot(lags, acf_teo, 'r--', lw=2, label='ACF teórica $\\phi^h$')
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline( 1.96/np.sqrt(n), color='gray', lw=1, ls=':')
    ax.axhline(-1.96/np.sqrt(n), color='gray', lw=1, ls=':')
    ax.set_title(f'n = {n}  ({n_rep} réplicas)')
    ax.set_xlabel('Lag h'); ax.set_ylim(-0.5, 1.1)
    ax.legend(fontsize=8)

axes[0].set_ylabel('ACF')
fig.suptitle('Estimación de la ACF — AR(1) con φ=0.7\n'
             'Ref: Brockwell & Davis (1991), Prop. 7.2.1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Bloque 1 — Estacionariedad e identificación

### ACF y PACF como herramientas de identificación

La **función de autocorrelación parcial** (PACF) mide la correlación entre $X_t$ y $X_{t+h}$ eliminando el efecto de los lags intermedios:

$$\phi_{hh} = \text{Corr}(X_t - \hat{X}_t,\; X_{t+h} - \hat{X}_{t+h})$$

donde $\hat{X}_t$ y $\hat{X}_{t+h}$ son las proyecciones sobre $\{X_{t+1}, \ldots, X_{t+h-1}\}$.

La PACF se obtiene resolviendo las **ecuaciones de Yule-Walker** para cada orden $h$.

| Modelo | ACF | PACF |
|--------|-----|------|
| AR(p)  | Decae (exponencial u oscilante) | Corta en lag p |
| MA(q)  | Corta en lag q | Decae |
| ARMA(p,q) | Decae | Decae |

In [ ]:
# ACF y PACF teóricas y empíricas para AR(1), AR(2), MA(1), MA(2)
n_sim = 1000
max_lag_id = 20

modelos = [
    ('AR(1)  φ=0.8',    ArmaProcess(ar=[1, -0.8],       ma=[1])),
    ('AR(2)  φ₁=0.5, φ₂=0.3', ArmaProcess(ar=[1,-0.5,-0.3], ma=[1])),
    ('MA(1)  θ=0.7',    ArmaProcess(ar=[1],             ma=[1, 0.7])),
    ('MA(2)  θ₁=0.6, θ₂=−0.4', ArmaProcess(ar=[1],    ma=[1, 0.6, -0.4])),
]

fig, axes = plt.subplots(4, 2, figsize=(14, 14))

for row, (nombre, proceso) in enumerate(modelos):
    serie = proceso.generate_sample(nsample=n_sim)

    # ACF
    acf_vals  = acf(serie,  nlags=max_lag_id, fft=True, alpha=0.05)
    pacf_vals = pacf(serie, nlags=max_lag_id, alpha=0.05)

    lags_plot = np.arange(max_lag_id + 1)
    banda = 1.96 / np.sqrt(n_sim)

    for col, (vals, titulo_col) in enumerate([
        (acf_vals[0],  f'{nombre} — ACF'),
        (pacf_vals[0], f'{nombre} — PACF')
    ]):
        ax = axes[row, col]
        ax.bar(lags_plot, vals, color='#2196F3' if col==0 else '#E91E63',
               alpha=0.7, width=0.6)
        ax.axhline( banda, color='gray', lw=1, ls='--')
        ax.axhline(-banda, color='gray', lw=1, ls='--')
        ax.axhline(0, color='black', lw=0.8)
        ax.set_title(titulo_col, fontsize=10)
        ax.set_xlabel('Lag h')
        ax.set_ylim(-1.1, 1.1)

fig.suptitle('ACF y PACF empíricas — patrón de identificación\n'
             'Ref: Box et al. (2015), Time Series Analysis, Cap. 6',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Tests de raíz unitaria

Antes de ajustar un modelo ARMA es necesario verificar estacionariedad.  
Los dos tests más usados tienen hipótesis opuestas:

- **ADF** (Augmented Dickey-Fuller): $H_0$: raíz unitaria (no estacionario) vs $H_1$: estacionario.  
- **KPSS** (Kwiatkowski-Phillips-Schmidt-Shin): $H_0$: estacionario vs $H_1$: raíz unitaria.

Se recomienda aplicar ambos en conjunto: si ADF rechaza y KPSS no rechaza → evidencia de estacionariedad.

**Referencias:** Dickey & Fuller (1979), JASA 74. Kwiatkowski et al. (1992), Journal of Econometrics 54.

In [ ]:
# Comparación: serie estacionaria vs caminata aleatoria vs con tendencia
n_test = 300
eps_t = np.random.normal(0, 1, n_test)

# AR(1) estacionario
ar1_est = np.zeros(n_test)
for t in range(1, n_test):
    ar1_est[t] = 0.7 * ar1_est[t-1] + eps_t[t]

# Caminata aleatoria (raíz unitaria)
caminata = np.cumsum(eps_t)

# AR(1) con tendencia
ar1_tend = ar1_est + 0.08 * np.arange(n_test)

series_test = [
    ('AR(1) estacionario  φ=0.7',  ar1_est,  '#2196F3'),
    ('Caminata aleatoria',          caminata, '#E91E63'),
    ('AR(1) con tendencia',         ar1_tend, '#FF9800'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (nombre, serie, color) in zip(axes, series_test):
    ax.plot(serie, color=color, lw=1.2, alpha=0.85)

    # ADF
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(serie, autolag='AIC')
    # KPSS
    kpss_stat, kpss_p, _, kpss_crit = kpss(serie, regression='c', nlags='auto')

    adf_concl  = 'Estacionario' if adf_p  < 0.05 else 'Raíz unitaria'
    kpss_concl = 'Estacionario' if kpss_p > 0.05 else 'No estacionario'

    ax.set_title(f'{nombre}\n'
                 f'ADF p={adf_p:.3f} → {adf_concl}\n'
                 f'KPSS p={kpss_p:.3f} → {kpss_concl}', fontsize=9)
    ax.set_xlabel('t'); ax.set_ylabel('X_t')

fig.suptitle('Tests de raíz unitaria: ADF y KPSS\n'
             'Ref: Dickey & Fuller (1979), JASA 74  |  Kwiatkowski et al. (1992), J. Econometrics 54',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Bloque 2 — Modelos AR(p)

Un proceso **AR(p)** (autorregresivo de orden p) satisface:

$$X_t = \phi_1 X_{t-1} + \cdots + \phi_p X_{t-p} + W_t, \quad W_t \sim \text{iid}(0, \sigma^2)$$

En notación de operador de rezago $B$ ($BX_t = X_{t-1}$):

$$\Phi(B) X_t = W_t \quad \text{con} \quad \Phi(z) = 1 - \phi_1 z - \cdots - \phi_p z^p$$

**Condición de estacionariedad:** todas las raíces del polinomio $\Phi(z) = 0$ deben estar **fuera del círculo unitario** $|z| > 1$.  
Equivalentemente, los autovalores de la matriz compañera tienen módulo menor que 1.

Bajo estacionariedad, el AR(p) admite representación $\text{MA}(\infty)$:
$$X_t = \sum_{j=0}^\infty \psi_j W_{t-j}$$

**Referencia:** Brockwell & Davis (1991), Cap. 3.

In [ ]:
# Raíces del polinomio AR y región de estacionariedad
# Para AR(2): phi1, phi2 con región triangular

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: región de estacionariedad AR(2)
phi1_grid = np.linspace(-2.5, 2.5, 400)
phi2_grid = np.linspace(-1.5, 1.5, 400)
P1, P2 = np.meshgrid(phi1_grid, phi2_grid)

# Condiciones: phi2 < 1 + phi1, phi2 < 1 - phi1, phi2 > -1
estacionario = (P2 < 1 + P1) & (P2 < 1 - P1) & (P2 > -1)

axes[0].contourf(P1, P2, estacionario.astype(float), levels=[0.5, 1.5],
                 colors=['#2196F3'], alpha=0.3)
axes[0].contour(P1, P2, estacionario.astype(float), levels=[0.5],
                colors=['#2196F3'], linewidths=2)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].axvline(0, color='black', lw=0.8)

# Marcar algunos modelos
ejemplos_ar2 = [(0.5, 0.3, 'A'), (-0.5, 0.4, 'B'), (1.5, 0.2, 'C (no est.)')]
for p1, p2, label in ejemplos_ar2:
    color = '#4CAF50' if abs(p1) + abs(p2) < 1 else '#E91E63'
    axes[0].scatter(p1, p2, s=100, zorder=5, color=color)
    axes[0].annotate(label, (p1, p2), textcoords='offset points',
                     xytext=(8, 5), fontsize=10)

axes[0].set_xlabel('φ₁'); axes[0].set_ylabel('φ₂')
axes[0].set_title('Región de estacionariedad AR(2)\n(área azul = estacionario)')

# Panel 2: raíces en el plano complejo para distintos AR(2)
configs = [
    ([1, -0.5, -0.3], '#2196F3', 'φ₁=0.5, φ₂=0.3 (real)'),
    ([1, -0.3,  0.8], '#E91E63', 'φ₁=0.3, φ₂=−0.8 (complejas)'),
]
theta_circ = np.linspace(0, 2*np.pi, 300)
axes[1].plot(np.cos(theta_circ), np.sin(theta_circ), 'k--', lw=1, alpha=0.5)
axes[1].axhline(0, color='gray', lw=0.5); axes[1].axvline(0, color='gray', lw=0.5)

for ar_coef, color, label in configs:
    raices = np.roots(ar_coef)
    axes[1].scatter(raices.real, raices.imag, s=120, color=color, zorder=5,
                    edgecolors='black', lw=0.8, label=label)

axes[1].set_xlabel('Re(z)'); axes[1].set_ylabel('Im(z)')
axes[1].set_title('Raíces de Φ(z) en plano complejo\n(deben estar fuera del círculo unitario)')
axes[1].legend(fontsize=8); axes[1].set_aspect('equal')
axes[1].set_xlim(-3, 3); axes[1].set_ylim(-3, 3)

# Panel 3: trayectorias con distintos phi
n_ar = 200
configs_tray = [
    (0.9,  '#2196F3', 'φ=0.9 (persistente)'),
    (0.0,  '#4CAF50', 'φ=0 (ruido blanco)'),
    (-0.8, '#E91E63', 'φ=−0.8 (oscilante)'),
]
for phi, color, label in configs_tray:
    x = np.zeros(n_ar)
    for t in range(1, n_ar):
        x[t] = phi * x[t-1] + np.random.normal()
    axes[2].plot(x, lw=1.2, alpha=0.8, color=color, label=label)

axes[2].set_xlabel('t'); axes[2].set_ylabel('X_t')
axes[2].set_title('Trayectorias AR(1) con distintos φ')
axes[2].legend(fontsize=9)

fig.suptitle('Modelos AR — estacionariedad y estructura de trayectorias\n'
             'Ref: Brockwell & Davis (1991), Time Series: Theory and Methods, Cap. 3',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Estimación por Yule-Walker

Las **ecuaciones de Yule-Walker** expresan los coeficientes AR en términos de la ACF:

$$\begin{pmatrix} \rho(1) \\ \vdots \\ \rho(p) \end{pmatrix} = \begin{pmatrix} 1 & \rho(1) & \cdots & \rho(p-1) \\ \rho(1) & 1 & \cdots & \rho(p-2) \\ \vdots & & \ddots & \vdots \\ \rho(p-1) & \cdots & \rho(1) & 1 \end{pmatrix} \begin{pmatrix} \phi_1 \\ \vdots \\ \phi_p \end{pmatrix}$$

Sustituyendo $\rho(h)$ por $\hat{\rho}(h)$ se obtiene el estimador de Yule-Walker. Es consistente y asintóticamente normal, aunque menos eficiente que MV.

In [ ]:
# Yule-Walker implementado desde cero vs statsmodels
def yule_walker_manual(x, p):
    """Estimación AR(p) por Yule-Walker."""
    n = len(x)
    x_c = x - x.mean()
    # Autocovarianzas empíricas
    gamma = np.array([np.mean(x_c[:n-h] * x_c[h:]) for h in range(p+1)])
    rho = gamma / gamma[0]
    # Sistema de Toeplitz
    R = toeplitz(rho[:p])
    phi = np.linalg.solve(R, rho[1:p+1])
    sigma2 = gamma[0] * (1 - rho[1:p+1] @ phi)
    return phi, sigma2

# Datos: AR(2) conocido
phi_true = np.array([0.6, -0.3])
sigma_true = 1.0
n_yw = 500

x_ar2 = np.zeros(n_yw)
eps_yw = np.random.normal(0, sigma_true, n_yw)
for t in range(2, n_yw):
    x_ar2[t] = phi_true[0]*x_ar2[t-1] + phi_true[1]*x_ar2[t-2] + eps_yw[t]

phi_yw, sigma2_yw = yule_walker_manual(x_ar2, p=2)

# Comparar con MV via statsmodels
modelo_mv = ARIMA(x_ar2, order=(2,0,0)).fit()

print("Estimación AR(2)  —  φ₁=0.6, φ₂=−0.3, σ²=1.0")
print("-" * 55)
print(f"{'Método':<20} {'φ₁':>8} {'φ₂':>8} {'σ²':>8}")
print(f"{'Verdadero':<20} {phi_true[0]:>8.3f} {phi_true[1]:>8.3f} {sigma_true**2:>8.3f}")
print(f"{'Yule-Walker':<20} {phi_yw[0]:>8.3f} {phi_yw[1]:>8.3f} {sigma2_yw:>8.3f}")
print(f"{'Máx. Verosimilitud':<20} {modelo_mv.arparams[0]:>8.3f} {modelo_mv.arparams[1]:>8.3f} {modelo_mv.sigma2:>8.3f}")

# Distribución de estimadores (bootstrap)
n_boot = 500
phi1_yw_boot, phi2_yw_boot = [], []
phi1_mv_boot, phi2_mv_boot = [], []

for _ in range(n_boot):
    x_b = np.zeros(n_yw)
    eps_b = np.random.normal(0, sigma_true, n_yw)
    for t in range(2, n_yw):
        x_b[t] = phi_true[0]*x_b[t-1] + phi_true[1]*x_b[t-2] + eps_b[t]
    ph, _ = yule_walker_manual(x_b, 2)
    phi1_yw_boot.append(ph[0]); phi2_yw_boot.append(ph[1])
    m = ARIMA(x_b, order=(2,0,0)).fit()
    phi1_mv_boot.append(m.arparams[0]); phi2_mv_boot.append(m.arparams[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, yw_vals, mv_vals, phi_v, titulo in zip(
    axes,
    [phi1_yw_boot, phi2_yw_boot],
    [phi1_mv_boot, phi2_mv_boot],
    phi_true,
    ['φ₁', 'φ₂']
):
    ax.hist(yw_vals, bins=30, density=True, alpha=0.5, color='#2196F3', label='Yule-Walker')
    ax.hist(mv_vals, bins=30, density=True, alpha=0.5, color='#E91E63', label='Máx. Verosimilitud')
    ax.axvline(phi_v, color='black', lw=2, ls='--', label=f'Verdadero = {phi_v}')
    ax.set_title(f'Distribución de estimadores de {titulo}  (n={n_yw}, {n_boot} réplicas)')
    ax.set_xlabel(titulo); ax.legend(fontsize=9)

fig.suptitle('Yule-Walker vs Máxima Verosimilitud — AR(2)\n'
             'Ref: Brockwell & Davis (1991), Prop. 8.1.1',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Bloque 3 — Modelos MA(q)

Un proceso **MA(q)** (media móvil de orden q) es:

$$X_t = W_t + \theta_1 W_{t-1} + \cdots + \theta_q W_{t-q} = \Theta(B) W_t$$

con $\Theta(z) = 1 + \theta_1 z + \cdots + \theta_q z^q$.

Todo MA(q) es estacionario (es una combinación lineal finita de ruido blanco).  
**Condición de invertibilidad:** las raíces de $\Theta(z) = 0$ deben estar fuera del círculo unitario. Esto garantiza la representación $\text{AR}(\infty)$ y la unicidad de la representación MA.

La ACF teórica corta exactamente en lag $q$:
$$\rho(h) = \frac{\sum_{j=0}^{q-|h|} \theta_j \theta_{j+|h|}}{\sum_{j=0}^q \theta_j^2} \quad \text{para } |h| \leq q, \qquad \rho(h) = 0 \text{ para } |h| > q$$

**Referencia:** Brockwell & Davis (1991), Cap. 3.  Box et al. (2015), Cap. 4.

In [ ]:
# Invertibilidad: MA(1) con theta vs 1/theta tienen la misma ACF
# Solo el invertible (|theta| < 1) admite representación AR(inf) convergente

theta_inv  =  0.6   # invertible
theta_ninv =  1/0.6 # no invertible (misma ACF)
n_ma = 500

eps_ma = np.random.normal(0, 1, n_ma + 1)
ma_inv  = eps_ma[1:] + theta_inv  * eps_ma[:-1]
ma_ninv = eps_ma[1:] + theta_ninv * eps_ma[:-1]

max_lag_ma = 15

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Trayectorias
axes[0,0].plot(ma_inv,  color='#2196F3', lw=1, alpha=0.8)
axes[0,0].set_title(f'MA(1) invertible  θ={theta_inv}')
axes[1,0].plot(ma_ninv, color='#E91E63', lw=1, alpha=0.8)
axes[1,0].set_title(f'MA(1) NO invertible  θ={theta_ninv:.2f} = 1/0.6')

# ACF
for row, (serie, theta, color) in enumerate([
    (ma_inv,  theta_inv,  '#2196F3'),
    (ma_ninv, theta_ninv, '#E91E63')
]):
    acf_emp = acf(serie, nlags=max_lag_ma, fft=True)
    lags_ma = np.arange(max_lag_ma + 1)

    # ACF teórica (ambas tienen la misma)
    acf_teo_ma = np.zeros(max_lag_ma + 1)
    acf_teo_ma[0] = 1.0
    acf_teo_ma[1] = theta / (1 + theta**2)

    axes[row, 1].bar(lags_ma - 0.2, acf_teo_ma, width=0.35, alpha=0.8,
                     color='black', label='Teórica')
    axes[row, 1].bar(lags_ma + 0.2, acf_emp, width=0.35, alpha=0.6,
                     color=color, label='Empírica')
    axes[row, 1].axhline( 1.96/np.sqrt(n_ma), color='gray', lw=1, ls='--')
    axes[row, 1].axhline(-1.96/np.sqrt(n_ma), color='gray', lw=1, ls='--')
    axes[row, 1].set_title('ACF — idéntica en ambos modelos')
    axes[row, 1].set_ylim(-0.8, 1.1)
    axes[row, 1].legend(fontsize=9)

    # Representación AR(inf): pesos psi_j = (-theta)^j
    j_vals = np.arange(20)
    psi_j = (-theta) ** j_vals
    axes[row, 2].bar(j_vals, psi_j, color=color, alpha=0.7)
    axes[row, 2].axhline(0, color='black', lw=0.8)
    axes[row, 2].set_title('Pesos AR(∞): $\\pi_j = (-\\theta)^j$\n'
                            + ('Converge' if abs(theta) < 1 else '¡No converge!'))
    axes[row, 2].set_xlabel('j')

for ax in axes[:, 0]:
    ax.set_xlabel('t'); ax.set_ylabel('X_t')

fig.suptitle('Invertibilidad en MA(1): θ y 1/θ tienen la misma ACF\n'
             'Ref: Box et al. (2015), Cap. 4  |  Brockwell & Davis (1991), Prop. 3.1.1',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ACF teórica para distintos MA(q)
def acf_teorica_ma(thetas, max_lag):
    """ACF teórica de un MA(q)."""
    q = len(thetas)
    th = np.concatenate([[1], thetas])
    gamma0 = np.sum(th**2)
    acf_t = np.zeros(max_lag + 1)
    acf_t[0] = 1.0
    for h in range(1, min(q+1, max_lag+1)):
        acf_t[h] = np.sum(th[:q+1-h] * th[h:]) / gamma0
    return acf_t

configs_ma = [
    ([0.7],          '#2196F3', 'MA(1)  θ=0.7'),
    ([-0.7],         '#E91E63', 'MA(1)  θ=−0.7'),
    ([0.6, 0.4],     '#4CAF50', 'MA(2)  θ₁=0.6, θ₂=0.4'),
    ([0.6, -0.5],    '#FF9800', 'MA(2)  θ₁=0.6, θ₂=−0.5'),
]

max_lag_teo = 12
lags_teo = np.arange(max_lag_teo + 1)

fig, axes = plt.subplots(2, 4, figsize=(18, 7))

for col, (thetas, color, nombre) in enumerate(configs_ma):
    proceso = ArmaProcess(ar=[1], ma=np.concatenate([[1], thetas]))
    serie = proceso.generate_sample(nsample=800)

    acf_teo_v  = acf_teorica_ma(thetas, max_lag_teo)
    acf_emp_v  = acf(serie, nlags=max_lag_teo, fft=True)

    axes[0, col].plot(serie[:150], color=color, lw=1, alpha=0.8)
    axes[0, col].set_title(nombre, fontsize=10)
    axes[0, col].set_xlabel('t')

    axes[1, col].bar(lags_teo - 0.2, acf_teo_v, width=0.35, color='black',
                     alpha=0.8, label='Teórica')
    axes[1, col].bar(lags_teo + 0.2, acf_emp_v, width=0.35, color=color,
                     alpha=0.6, label='Empírica')
    axes[1, col].axhline( 1.96/np.sqrt(800), color='gray', lw=1, ls='--')
    axes[1, col].axhline(-1.96/np.sqrt(800), color='gray', lw=1, ls='--')
    axes[1, col].set_xlabel('Lag h')
    axes[1, col].set_ylim(-1.1, 1.1)
    axes[1, col].legend(fontsize=8)

axes[0, 0].set_ylabel('X_t')
axes[1, 0].set_ylabel('ACF')

fig.suptitle('ACF teórica vs empírica — modelos MA(q): la ACF corta en lag q\n'
             'Ref: Brockwell & Davis (1991), Cap. 3',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Bloque 4 — ARMA(p,q) y ARIMA(p,d,q)

Un proceso **ARMA(p,q)** combina componentes AR y MA:

$$\Phi(B) X_t = \Theta(B) W_t$$

- **Estacionario** si las raíces de $\Phi(z)=0$ están fuera del círculo unitario
- **Invertible** si las raíces de $\Theta(z)=0$ están fuera del círculo unitario
- Tanto la ACF como la PACF **decaen** (no cortan): esto complica la identificación

Cuando la serie tiene raíz unitaria ($d$ raíces en $|z|=1$), se aplican $d$ diferencias para inducir estacionariedad, obteniendo un modelo **ARIMA(p,d,q)**:

$$\Phi(B)(1-B)^d X_t = \Theta(B) W_t$$

Para series con estacionalidad de período $s$ se agrega una componente estacional, dando lugar al **SARIMA$(p,d,q)(P,D,Q)_s$** — que no desarrollamos aquí.

**Referencia:** Box et al. (2015), Cap. 6. Hamilton (1994), Cap. 3.

In [ ]:
# ARMA: ACF y PACF decaen ambas
# Comparación ARMA(1,1), ARMA(2,1), ARMA(1,2)

configs_arma = [
    ([1, -0.7], [1,  0.5], 'ARMA(1,1)  φ=0.7, θ=0.5',  '#2196F3'),
    ([1, -0.6, -0.3], [1, 0.4], 'ARMA(2,1)  φ₁=0.6, φ₂=0.3, θ=0.4', '#4CAF50'),
    ([1, -0.5], [1, 0.6, -0.3], 'ARMA(1,2)  φ=0.5, θ₁=0.6, θ₂=−0.3', '#E91E63'),
]

n_arma = 800
max_lag_arma = 20

fig, axes = plt.subplots(3, 3, figsize=(16, 11))

for row, (ar_coef, ma_coef, nombre, color) in enumerate(configs_arma):
    proceso = ArmaProcess(ar=ar_coef, ma=ma_coef)
    serie = proceso.generate_sample(nsample=n_arma)

    acf_v  = acf(serie,  nlags=max_lag_arma, fft=True)
    pacf_v = pacf(serie, nlags=max_lag_arma)
    lags_a = np.arange(max_lag_arma + 1)
    banda  = 1.96 / np.sqrt(n_arma)

    axes[row, 0].plot(serie[:200], color=color, lw=1, alpha=0.85)
    axes[row, 0].set_title(nombre, fontsize=10)
    axes[row, 0].set_xlabel('t'); axes[row, 0].set_ylabel('X_t')

    for col_idx, (vals, titulo_col) in enumerate([
        (acf_v,  'ACF — decae'),
        (pacf_v, 'PACF — decae')
    ]):
        axes[row, col_idx+1].bar(lags_a, vals, color=color, alpha=0.7, width=0.6)
        axes[row, col_idx+1].axhline( banda, color='gray', lw=1, ls='--')
        axes[row, col_idx+1].axhline(-banda, color='gray', lw=1, ls='--')
        axes[row, col_idx+1].axhline(0, color='black', lw=0.8)
        axes[row, col_idx+1].set_title(titulo_col, fontsize=10)
        axes[row, col_idx+1].set_xlabel('Lag h')
        axes[row, col_idx+1].set_ylim(-1.1, 1.1)

fig.suptitle('ARMA(p,q) — ACF y PACF decaen ambas\n'
             'Ref: Box et al. (2015), Cap. 6  |  Hamilton (1994), Cap. 3',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ARIMA: diferenciación para inducir estacionariedad
# Ejemplo: caminata aleatoria con deriva -> ARIMA(0,1,0)
# y AR(1) integrado -> ARIMA(1,1,0)

n_arima = 300
eps_arima = np.random.normal(0, 1, n_arima)
deriva = 0.05

# ARIMA(0,1,0): caminata con deriva
x_arima010 = np.cumsum(deriva + eps_arima)
dx_010     = np.diff(x_arima010)  # después de diferenciar: ruido blanco con media

# ARIMA(1,1,0): AR(1) sobre la primera diferencia
phi_arima = 0.6
dx_110 = np.zeros(n_arima)
for t in range(1, n_arima):
    dx_110[t] = phi_arima * dx_110[t-1] + eps_arima[t]
x_arima110 = np.cumsum(dx_110)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for row, (x_orig, dx, nombre, color) in enumerate([
    (x_arima010, dx_010, 'ARIMA(0,1,0) — caminata con deriva', '#9C27B0'),
    (x_arima110, np.diff(x_arima110), 'ARIMA(1,1,0) — AR(1) integrado', '#00BCD4'),
]):
    axes[row, 0].plot(x_orig, color=color, lw=1.5, alpha=0.85)
    axes[row, 0].set_title(f'{nombre}\n(serie original — no estacionaria)')
    axes[row, 0].set_xlabel('t'); axes[row, 0].set_ylabel('X_t')

    axes[row, 1].plot(dx, color=color, lw=1, alpha=0.8)
    axes[row, 1].axhline(np.mean(dx), color='black', lw=1.5, ls='--',
                         label=f'Media = {np.mean(dx):.3f}')
    axes[row, 1].set_title('Primera diferencia ΔX_t\n(estacionaria)')
    axes[row, 1].set_xlabel('t'); axes[row, 1].legend(fontsize=9)

    acf_dx = acf(dx, nlags=20, fft=True)
    banda_a = 1.96 / np.sqrt(len(dx))
    axes[row, 2].bar(range(21), acf_dx, color=color, alpha=0.7, width=0.6)
    axes[row, 2].axhline( banda_a, color='gray', lw=1, ls='--')
    axes[row, 2].axhline(-banda_a, color='gray', lw=1, ls='--')
    axes[row, 2].axhline(0, color='black', lw=0.8)
    axes[row, 2].set_title('ACF de ΔX_t')
    axes[row, 2].set_xlabel('Lag h')
    axes[row, 2].set_ylim(-1.1, 1.1)

fig.suptitle('ARIMA — diferenciación para inducir estacionariedad\n'
             'Ref: Box et al. (2015), Cap. 4  |  Hamilton (1994), Cap. 15',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Bloque 5 — Selección de modelo y diagnóstico

### Criterios de información

Dado un modelo ARMA(p,q) ajustado por máxima verosimilitud con log-verosimilitud $\ell$:

$$\text{AIC} = -2\ell + 2k \qquad \text{BIC} = -2\ell + k\log T$$

donde $k = p + q + 1$ es el número de parámetros y $T$ el tamaño muestral.  
Se elige el modelo con menor AIC o BIC.  
BIC penaliza más fuerte y tiende a seleccionar modelos más parsimoniosos (consistente en selección de orden).  
AIC es asintóticamente eficiente pero no consistente.

**Referencia:** Akaike (1974), IEEE TAC 19.  Schwarz (1978), Annals of Statistics 6.

In [ ]:
# Selección de orden por AIC y BIC en una grilla de modelos ARMA(p,q)
# Datos generados por ARMA(2,1) verdadero

ar_verd = [1, -0.6, -0.3]
ma_verd = [1,  0.5]
proceso_verd = ArmaProcess(ar=ar_verd, ma=ma_verd)
n_sel = 300
serie_sel = proceso_verd.generate_sample(nsample=n_sel)

p_max = 4
q_max = 4

aic_grid = np.full((p_max+1, q_max+1), np.nan)
bic_grid = np.full((p_max+1, q_max+1), np.nan)

for p in range(p_max+1):
    for q in range(q_max+1):
        if p == 0 and q == 0:
            continue
        try:
            m = ARIMA(serie_sel, order=(p, 0, q)).fit()
            aic_grid[p, q] = m.aic
            bic_grid[p, q] = m.bic
        except:
            pass

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, grid, titulo, cmap in zip(
    axes, [aic_grid, bic_grid],
    ['AIC', 'BIC'],
    ['YlOrRd', 'YlOrRd']
):
    im = ax.imshow(grid, aspect='auto', cmap=cmap,
                   vmin=np.nanmin(grid), vmax=np.nanpercentile(grid, 80))
    plt.colorbar(im, ax=ax)

    # Marcar mínimo
    min_idx = np.unravel_index(np.nanargmin(grid), grid.shape)
    ax.scatter(min_idx[1], min_idx[0], marker='*', s=300,
               color='white', zorder=5, label=f'Mínimo: ARMA({min_idx[0]},{min_idx[1]})')

    # Marcar verdadero
    ax.scatter(1, 2, marker='o', s=200, color='cyan', zorder=5,
               edgecolors='black', lw=1.5, label='Verdadero: ARMA(2,1)')

    # Valores en celdas
    for i in range(p_max+1):
        for j in range(q_max+1):
            if not np.isnan(grid[i,j]):
                ax.text(j, i, f'{grid[i,j]:.0f}', ha='center', va='center',
                        fontsize=7.5, color='black')

    ax.set_xticks(range(q_max+1)); ax.set_xticklabels(range(q_max+1))
    ax.set_yticks(range(p_max+1)); ax.set_yticklabels(range(p_max+1))
    ax.set_xlabel('q  (orden MA)'); ax.set_ylabel('p  (orden AR)')
    ax.set_title(f'Grilla de {titulo}  (menor = mejor)\n'
                 f'Datos: ARMA(2,1) verdadero  —  n={n_sel}')
    ax.legend(fontsize=9, loc='upper right')

fig.suptitle('Selección de orden ARMA por AIC y BIC\n'
             'Ref: Akaike (1974), IEEE TAC 19  |  Schwarz (1978), Ann. Stat. 6',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Diagnóstico de residuos del modelo seleccionado
# Test de Ljung-Box: H0: los residuos son ruido blanco
# Referencia: Ljung & Box (1978), Biometrika 65

modelo_final = ARIMA(serie_sel, order=(2, 0, 1)).fit()
residuos = modelo_final.resid

# Test de Ljung-Box
lb_result = acorr_ljungbox(residuos, lags=range(1, 21), return_df=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Residuos en el tiempo
axes[0,0].plot(residuos, color='#607D8B', lw=1, alpha=0.8)
axes[0,0].axhline(0, color='black', lw=1, ls='--')
axes[0,0].set_title('Residuos en el tiempo')
axes[0,0].set_xlabel('t'); axes[0,0].set_ylabel('ê_t')

# ACF de residuos
acf_res = acf(residuos, nlags=20, fft=True)
banda_res = 1.96 / np.sqrt(n_sel)
axes[0,1].bar(range(21), acf_res, color='#607D8B', alpha=0.7, width=0.6)
axes[0,1].axhline( banda_res, color='gray', lw=1, ls='--')
axes[0,1].axhline(-banda_res, color='gray', lw=1, ls='--')
axes[0,1].axhline(0, color='black', lw=0.8)
axes[0,1].set_title('ACF de residuos\n(debe ser ruido blanco)')
axes[0,1].set_xlabel('Lag h'); axes[0,1].set_ylim(-1.1, 1.1)

# Test de Ljung-Box: p-values
axes[0,2].bar(lb_result.index, lb_result['lb_pvalue'],
              color=np.where(lb_result['lb_pvalue'] > 0.05, '#4CAF50', '#E91E63'),
              alpha=0.8, width=0.6)
axes[0,2].axhline(0.05, color='black', lw=2, ls='--', label='α=0.05')
axes[0,2].set_title('Test de Ljung-Box\np-valores por lag')
axes[0,2].set_xlabel('Lag h'); axes[0,2].set_ylabel('p-valor')
axes[0,2].legend(fontsize=9)
axes[0,2].set_ylim(0, 1)

# Distribución de residuos
axes[1,0].hist(residuos, bins=35, density=True,
               color='#607D8B', alpha=0.6, label='Empírica')
xx_r = np.linspace(residuos.min(), residuos.max(), 200)
mu_r, sig_r = residuos.mean(), residuos.std()
axes[1,0].plot(xx_r, stats.norm.pdf(xx_r, mu_r, sig_r), 'r-', lw=2, label='N(μ̂,σ̂²)')
axes[1,0].set_title('Distribución de residuos')
axes[1,0].set_xlabel('ê_t'); axes[1,0].legend(fontsize=9)

# QQ-plot
stats.probplot(residuos, dist='norm', plot=axes[1,1])
axes[1,1].set_title('QQ-plot de residuos')
axes[1,1].get_lines()[0].set(color='#607D8B', alpha=0.6, markersize=3)
axes[1,1].get_lines()[1].set(color='red', lw=2)

# Resumen del modelo
axes[1,2].axis('off')
resumen = (
    f"Modelo ajustado: ARMA(2,1)\n\n"
    f"φ₁ = {modelo_final.arparams[0]:.3f}  (verdadero: 0.600)\n"
    f"φ₂ = {modelo_final.arparams[1]:.3f}  (verdadero: 0.300)\n"
    f"θ₁ = {modelo_final.maparams[0]:.3f}  (verdadero: 0.500)\n"
    f"σ² = {modelo_final.sigma2:.3f}  (verdadero: 1.000)\n\n"
    f"AIC = {modelo_final.aic:.1f}\n"
    f"BIC = {modelo_final.bic:.1f}\n\n"
    f"Ljung-Box lag 10: p = {lb_result.loc[10,'lb_pvalue']:.3f}\n"
    f"  → {'No se rechaza H₀ (residuos OK)' if lb_result.loc[10,'lb_pvalue'] > 0.05 else 'Se rechaza H₀'}"
)
axes[1,2].text(0.05, 0.95, resumen, transform=axes[1,2].transAxes,
               fontsize=11, va='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))

fig.suptitle('Diagnóstico de residuos — modelo ARMA(2,1)\n'
             'Ref: Ljung & Box (1978), Biometrika 65(2)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(modelo_final.summary())